In [9]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *
cdo = Cdo()

import shutil
import tempfile
from pathlib import Path

In [ ]:
test = xc.open_dataset('../data/CNRM-CM6-1_amip-hist_N_128x64_200003-201412/N_Amon_CNRM-CM6-1_amip-hist_members_128x64_187001-201412.nc')

In [ ]:
test['N'][0][0].plot()

In [15]:
# Cell 1 — Imports and configuration

from pathlib import Path
from collections import defaultdict
import csv
import re
import subprocess
import tempfile

import numpy as np
from netCDF4 import Dataset


# Maria's archive root.
maria_root = Path("/rugenstein-archive/mariarug/model_output/MMLEA")

# Target 128x64 grid.
# lowres_template = Path("/scratch/leiff/amip/data/lowres_template.nc")

# Going to target Maria's ETH resolution instead now
g025_template = Path("/scratch/leiff/MCA/data/rlut_mon_CanESM5_historical_r1i1p1f1_g025.nc")
lowres_template = []# emptying out to force error if old one got left in anywhere

# Use the detailed CSV rather than maria_processing_tasks.json:
# it includes both ordinary one-file records and time-split ("ambiguous") records.
catalog_path = Path("./maria_catalog/maria_catalog.csv")

# Final output layout:
# ./data/AMIP/<model>/N_Amon_<model>_<experiment>_combined_187001-201412.nc
output_root = Path("./data/AMIP")

# All output files are restricted to this common period.
start_date = "1870-01-01"
end_date = "2014-12-31"
output_period = "187001-201412"

# Leave False for now: finished combined files will be skipped rather than replaced.
overwrite = False

In [3]:
# Cell 2 — Senne overlap and Maria task selection
# Models for which Senne has processed monthly tas.
# This determines the requested overlap; Maria supplies rsdt, rlut, and rsut.

senne_tas_models = {
    "amip-hist": {
        "BCC-CSM2-MR",
        "CAMS-CSM1-0",
        "CESM2",
        "CIESM",
        "CNRM-CM6-1",
        "CNRM-CM6-1-HR",
        "CNRM-ESM2-1",
        "CanESM5",
        "FGOALS-f3-L",
        "FGOALS-g3",
        "FIO-ESM-2-0",
        "IITM-ESM",
        "IPSL-CM6A-LR",
        "MIROC6",
        "MRI-ESM2-0",
        "TaiESM1",
    },
    "amip-piForcing": {
        "CESM2",
        "CNRM-CM6-1",
        "CanESM5",
        "HadGEM3-GC31-LL",
        "IPSL-CM6A-LR",
        "MIROC6",
        "MRI-ESM2-0",
        "TaiESM1",
    },
}

radiation_variables = ("rsdt", "rlut", "rsut")

with catalog_path.open(newline="") as handle:
    maria_catalog = list(csv.DictReader(handle))

# Keep a member if Maria has all three radiation components.
# We intentionally do NOT require Maria tas here: Senne's processed tas is the
# matching tas product, while Maria is only needed to make spatial N.
tasks_by_model_experiment = defaultdict(list)

for row in maria_catalog:
    model = row["model"]
    experiment = row["experiment"]

    if experiment not in senne_tas_models:
        continue
    if model not in senne_tas_models[experiment]:
        continue
    if any(not row[f"{variable}_path"] for variable in radiation_variables):
        continue

    tasks_by_model_experiment[(model, experiment)].append(row)

print("Maria radiation-data overlap with Senne's processed monthly tas:")
for (model, experiment), tasks in sorted(tasks_by_model_experiment.items()):
    print(f"  {experiment:15s}  {model:20s}  {len(tasks)} member(s)")

Maria radiation-data overlap with Senne's processed monthly tas:
  amip-hist        BCC-CSM2-MR           1 member(s)
  amip-hist        CAMS-CSM1-0           3 member(s)
  amip-hist        CESM2                 3 member(s)
  amip-piForcing   CESM2                 1 member(s)
  amip-hist        CIESM                 3 member(s)
  amip-hist        CNRM-CM6-1            10 member(s)
  amip-piForcing   CNRM-CM6-1            1 member(s)
  amip-hist        CNRM-CM6-1-HR         1 member(s)
  amip-hist        CNRM-ESM2-1           1 member(s)
  amip-hist        CanESM5               10 member(s)
  amip-piForcing   CanESM5               3 member(s)
  amip-hist        FGOALS-f3-L           3 member(s)
  amip-hist        FGOALS-g3             4 member(s)
  amip-hist        FIO-ESM-2-0           3 member(s)
  amip-piForcing   HadGEM3-GC31-LL       1 member(s)
  amip-hist        IITM-ESM              1 member(s)
  amip-hist        IPSL-CM6A-LR          3 member(s)
  amip-piForcing   IPSL-CM6A-LR 

In [11]:
# Cell 3 — Helpers for time chunks and CDO processing

def realization_number(member_name):
    """Numerical ordering: r1, r2, ..., r10 rather than alphabetical order."""
    return int(re.match(r"r(\d+)", member_name).group(1))

def cdo_input_stream(paths):
    """
    Wrap each variable stream in CDO argument-group brackets.

    This lets CDO distinguish:
      merge(mergetime(rsdt chunks),
            mergetime(rlut chunks),
            mergetime(rsut chunks))
    """
    if len(paths) == 1:
        return ["[", str(paths[0]), "]"]

    return [
        "[",
        "-mergetime",
        *map(str, paths),
        "]",
    ]

def month_number(yyyymm):
    """Convert YYYYMM to a monotonically increasing month number."""
    year = int(yyyymm[:4])
    month = int(yyyymm[4:6])
    return year * 12 + month - 1


def member_paths(row, variable):
    """
    Return a variable's source files in chronological order.

    A normal Maria record has one path.
    A catalog 'ambiguous' record contains semicolon-separated, sequential
    time chunks. These are safe to use when they form one continuous series.
    """
    relative_paths = [
        path for path in row[f"{variable}_path"].split(";")
        if path
    ]

    paths_with_periods = []
    for relative_path in relative_paths:
        match = re.search(r"_(\d{6})-(\d{6})\.nc$", relative_path)
        if not match:
            raise ValueError(f"Could not read date range from {relative_path}")

        start, end = match.groups()
        paths_with_periods.append(
            (month_number(start), month_number(end), maria_root / relative_path.lstrip("./"))
        )

    paths_with_periods.sort()

    # Confirm that adjacent source files are consecutive monthly chunks,
    # rather than duplicate versions or competing grids.
    for (_, previous_end, _), (next_start, _, _) in zip(
        paths_with_periods[:-1],
        paths_with_periods[1:],
    ):
        if next_start != previous_end + 1:
            raise ValueError(
                f"{row['model']} {row['experiment']} {row['member']} {variable}: "
                "source chunks are not continuous in time."
            )

    # Every source collection must cover the desired common output period.
    if paths_with_periods[0][0] > month_number("187001"):
        raise ValueError(f"{variable} starts after 1870-01 for {row['member']}")
    if paths_with_periods[-1][1] < month_number("201412"):
        raise ValueError(f"{variable} ends before 2014-12 for {row['member']}")

    paths = [path for _, _, path in paths_with_periods]

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing Maria source file(s):\n" + "\n".join(missing))

    return paths


def create_member_n(row, temporary_n_file):
    """
    Merge sequential time chunks for rsdt, rlut, and rsut,
    calculate N = rsdt - rlut - rsut,
    clip to 1870–2014,
    and conservatively regrid to the low-resolution template.
    """
    rsdt_paths = member_paths(row, "rsdt")
    rlut_paths = member_paths(row, "rlut")
    rsut_paths = member_paths(row, "rsut")

    # Temporary files for the three merged variables
    tmp_dir = Path(tempfile.mkdtemp(prefix="cdo_inputs_"))

    try:
        rsdt_merged = tmp_dir / "rsdt.nc"
        rlut_merged = tmp_dir / "rlut.nc"
        rsut_merged = tmp_dir / "rsut.nc"
        merged = tmp_dir / "merged.nc"

        def mergetime(paths, output):
            command = [
                "cdo",
                "-L",
                "-O",
                "mergetime",
                *map(str, paths),
                str(output),
            ]

            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
            )

            if result.returncode != 0:
                print("\nCDO command:")
                print(" ".join(command))
                print("\nCDO stderr:")
                print(result.stderr)
                raise RuntimeError(
                    f"CDO mergetime failed for {row['model']} / "
                    f"{row['experiment']} / {row['member']}"
                )

        # Merge each variable's time chunks
        mergetime(rsdt_paths, rsdt_merged)
        mergetime(rlut_paths, rlut_merged)
        mergetime(rsut_paths, rsut_merged)

        # Put the three variables into one file
        command = [
            "cdo",
            "-L",
            "-O",
            "merge",
            str(rsdt_merged),
            str(rlut_merged),
            str(rsut_merged),
            str(merged),
        ]

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO merge failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

        # Calculate N, select dates, and remap
        command = [
            "cdo",
            "-L",
            "-O",
            "-f", "nc4c",
            "-z", "zip_4",
            f"remapcon,{g025_template}",
            f"-seldate,{start_date},{end_date}",
            "-expr,N=rsdt-rlut-rsut",
            str(merged),
            str(temporary_n_file),
        ]

        print(f"    CDO: {row['member']}")

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

In [5]:
# Cell 4 — Write members one at a time into a combined NetCDF

def copy_member_to_combined(source_file, combined_file, member_index, member_name):
    """
    Copy a single temporary N field into its proper location in the final file.

    The final file contains only:
      - member(member): CMIP6 strings, such as r1i1p1f2
      - time, lat, lon
      - N(member, time, lat, lon)

    This avoids keeping a model's entire ensemble in memory.
    """
    with Dataset(source_file, "r") as source:
        source_n = source.variables["N"]
        source_dimensions = source_n.dimensions

        # The first member creates the final file and its shared coordinates.
        if not combined_file.exists():
            with Dataset(combined_file, "w", format="NETCDF4") as destination:
                destination.createDimension("member", None)

                for dimension in source_dimensions:
                    destination.createDimension(
                        dimension,
                        len(source.dimensions[dimension]),
                    )

                # Copy coordinate variables, normally time, lat, and lon.
                for dimension in source_dimensions:
                    source_coordinate = source.variables[dimension]

                    destination_coordinate = destination.createVariable(
                        dimension,
                        source_coordinate.dtype,
                        (dimension,),
                    )
                    destination_coordinate[:] = source_coordinate[:]

                    for attribute in source_coordinate.ncattrs():
                        destination_coordinate.setncattr(
                            attribute,
                            source_coordinate.getncattr(attribute),
                        )

                # The named coordinate that Senne's files lack.
                member_coordinate = destination.createVariable(
                    "member",
                    str,
                    ("member",),
                )
                member_coordinate.long_name = "CMIP6 ensemble member identifier"
                member_coordinate.comment = (
                    "Member order follows the numerical CMIP realization number."
                )

                # Write N compressed, with chunks that keep one member and one
                # year of monthly data together.
                n_output = destination.createVariable(
                    "N",
                    "f4",
                    ("member",) + source_dimensions,
                    zlib=True,
                    complevel=4,
                    chunksizes=(
                        1,
                        min(12, len(source.dimensions["time"])),
                        len(source.dimensions["lat"]),
                        len(source.dimensions["lon"]),
                    ),
                    fill_value=np.float32(np.nan),
                )

                for attribute in source_n.ncattrs():
                    if attribute != "_FillValue":
                        n_output.setncattr(
                            attribute,
                            source_n.getncattr(attribute),
                        )

                n_output.long_name = "Net top-of-atmosphere radiation"
                n_output.N_definition = "rsdt - rlut - rsut"
                n_output.regridding = "CDO remapcon to lowres_template.nc"

        # Add this member, then close the temporary source file before moving on.
        with Dataset(combined_file, "a") as destination:
            destination.variables["member"][member_index] = member_name
            destination.variables["N"][member_index, :, :, :] = source_n[:, :, :]

In [14]:
# Cell 5 — Run every selected model and experiment
for (model, experiment), tasks in sorted(tasks_by_model_experiment.items()):
    # Use Senne's numerical ensemble ordering.
    tasks = sorted(tasks, key=lambda row: realization_number(row["member"]))

    model_output_dir = output_root / model
    model_output_dir.mkdir(parents=True, exist_ok=True)

    final_file = (
        model_output_dir
        / f"N_Amon_{model}_{experiment}_combined_{output_period}_g025.nc"
    )
    partial_file = final_file.with_suffix(".partial.nc")

    # Do not accidentally replace a finished result.
    if final_file.exists() and not overwrite:
        print(f"Skipping existing file: {final_file}")
        continue

    # A partial file is only used while this model/experiment is processing.
    # It is renamed to the final filename only after every member succeeds.
    if partial_file.exists():
        partial_file.unlink()

    print(f"\n{model} / {experiment}: {len(tasks)} member(s)")

    try:
        with tempfile.TemporaryDirectory(prefix="cdo_N_") as temporary_directory:
            temporary_directory = Path(temporary_directory)

            for member_index, row in enumerate(tasks):
                temporary_n_file = temporary_directory / f"{row['member']}_N.nc"

                # Create N for one member, copy it to the combined file, and
                # delete the temporary file before processing the next member.
                create_member_n(row, temporary_n_file)

                copy_member_to_combined(
                    temporary_n_file,
                    partial_file,
                    member_index,
                    row["member"],
                )

                temporary_n_file.unlink()

        # This rename marks the product as complete.
        partial_file.replace(final_file)

        print(f"  Saved: {final_file}")
        subprocess.run(["du", "-h", str(final_file)], check=True)

    except Exception:
        print(f"  FAILED: incomplete output retained at {partial_file}")
        raise



BCC-CSM2-MR / amip-hist: 1 member(s)


    CDO: r1i1p1f1
  Saved: data/AMIP/BCC-CSM2-MR/N_Amon_BCC-CSM2-MR_amip-hist_combined_187001-201412_g025.nc
512	data/AMIP/BCC-CSM2-MR/N_Amon_BCC-CSM2-MR_amip-hist_combined_187001-201412_g025.nc

CAMS-CSM1-0 / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CAMS-CSM1-0/N_Amon_CAMS-CSM1-0_amip-hist_combined_187001-201412_g025.nc

CESM2 / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CESM2/N_Amon_CESM2_amip-hist_combined_187001-201412_g025.nc

CESM2 / amip-piForcing: 1 member(s)
    CDO: r1i1p1f1
  Saved: data/AMIP/CESM2/N_Amon_CESM2_amip-piForcing_combined_187001-201412_g025.nc

CIESM / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CIESM/N_Amon_CIESM_amip-hist_combined_187001-201412_g025.nc

CNRM-CM6-1 / amip-hist: 10 member(s)
    CDO: r1i1p1f2
    CDO: r2i1p1f2
    CDO: r3i1p1f2
    CDO: r4i1p1f2
    CDO: r5i1p1f2
    CDO: r6i1p1f2
    CDO

In [16]:
# Build tas jobs from the actual completed g025 N outputs.
#
# For every labeled member in an N file, require the same model, experiment,
# and CMIP6 member in Maria's catalog, with continuous tas coverage spanning
# 1870-01 through 2014-12.

tas_remap_operator = "remapcon"

# Leave False unless replacing tas outputs created during an earlier attempt.
tas_overwrite = False


def decode_member_label(value):
    """Convert NetCDF byte strings into normal Python strings."""
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def maria_tas_paths(row):
    """
    Return Maria's original tas files for one member in chronological order.

    Semicolon-separated catalog paths are treated as sequential time chunks,
    just as they are for rsdt, rlut, and rsut in the N pipeline.
    """
    raw_paths = row.get("tas_path", "")

    if not isinstance(raw_paths, str) or not raw_paths.strip():
        raise ValueError(
            f"No Maria tas files for "
            f"{row['model']} / {row['experiment']} / {row['member']}"
        )

    relative_paths = [
        path.strip()
        for path in raw_paths.split(";")
        if path.strip()
    ]

    if not relative_paths:
        raise ValueError(
            f"No Maria tas files for "
            f"{row['model']} / {row['experiment']} / {row['member']}"
        )

    paths_with_periods = []

    for relative_path in relative_paths:
        period_match = re.search(
            r"_(\d{6})-(\d{6})\.nc$",
            relative_path,
        )

        if period_match is None:
            raise ValueError(
                f"Could not read a monthly date range from "
                f"{relative_path}"
            )

        first_month, last_month = period_match.groups()

        absolute_path = (
            maria_root / relative_path.lstrip("./")
        )

        paths_with_periods.append(
            (
                month_number(first_month),
                month_number(last_month),
                absolute_path,
            )
        )

    paths_with_periods.sort()

    # Accept sequential chunks but reject gaps, overlaps, and competing files.
    for (
        (_, previous_end, _),
        (next_start, _, _),
    ) in zip(
        paths_with_periods[:-1],
        paths_with_periods[1:],
    ):
        if next_start != previous_end + 1:
            raise ValueError(
                f"{row['model']} / {row['experiment']} / "
                f"{row['member']}: tas chunks are not continuous"
            )

    if (
        paths_with_periods[0][0]
        > month_number("187001")
    ):
        raise ValueError(
            f"tas starts after 1870-01 for "
            f"{row['model']} / {row['experiment']} / {row['member']}"
        )

    if (
        paths_with_periods[-1][1]
        < month_number("201412")
    ):
        raise ValueError(
            f"tas ends before 2014-12 for "
            f"{row['model']} / {row['experiment']} / {row['member']}"
        )

    paths = [
        path
        for _, _, path in paths_with_periods
    ]

    missing_files = [
        str(path)
        for path in paths
        if not path.exists()
    ]

    if missing_files:
        raise FileNotFoundError(
            "Missing Maria tas source file(s):\n"
            + "\n".join(missing_files)
        )

    return paths


# The catalog is already loaded as a list of dictionaries in the N pipeline.
catalog_rows_by_member = defaultdict(list)

for row in maria_catalog:
    key = (
        row["model"],
        row["experiment"],
        row["member"],
    )
    catalog_rows_by_member[key].append(row)


n_filename_pattern = re.compile(
    r"^N_Amon_"
    r"(?P<model>.+)_"
    r"(?P<experiment>amip-hist|amip-piForcing)_"
    r"combined_187001-201412_g025\.nc$"
)

completed_n_files = sorted(
    output_root.glob(
        "*/N_Amon_*_combined_187001-201412_g025.nc"
    )
)

if not completed_n_files:
    raise FileNotFoundError(
        f"No completed g025 N files found under {output_root}"
    )

maria_tas_tasks = []
tas_inventory_problems = []

print(
    "Maria original tas matched to completed g025 N outputs:"
)

for n_file in completed_n_files:
    filename_match = n_filename_pattern.match(n_file.name)

    if filename_match is None:
        continue

    model = filename_match.group("model")
    experiment = filename_match.group("experiment")

    # These are the exact labels and ordering required in the tas output.
    with Dataset(n_file, "r") as n_dataset:
        n_member_labels = [
            decode_member_label(value)
            for value in n_dataset.variables["member"][:]
        ]

    ordered_catalog_rows = []
    member_problems = []

    for member_label in n_member_labels:
        key = (model, experiment, member_label)
        matching_rows = catalog_rows_by_member.get(key, [])

        if not matching_rows:
            member_problems.append(
                f"{member_label}: absent from Maria catalog"
            )
            continue

        if len(matching_rows) > 1:
            member_problems.append(
                f"{member_label}: {len(matching_rows)} catalog rows"
            )
            continue

        row = matching_rows[0]

        try:
            # This validates paths, chunks, continuity, and coverage now.
            maria_tas_paths(row)
            ordered_catalog_rows.append(row)

        except Exception as exc:
            member_problems.append(
                f"{member_label}: {exc}"
            )

    if member_problems:
        tas_inventory_problems.append(
            {
                "model": model,
                "experiment": experiment,
                "problems": member_problems,
            }
        )

        print(
            f"  {experiment:<16}{model:<22}"
            f"{len(ordered_catalog_rows):>2}/"
            f"{len(n_member_labels):<2} tas member(s)  NOT READY"
        )

        for problem in member_problems:
            print(f"      {problem}")

    else:
        maria_tas_tasks.append(
            {
                "model": model,
                "experiment": experiment,
                "n_file": n_file,
                "member_labels": n_member_labels,
                "catalog_rows": ordered_catalog_rows,
            }
        )

        print(
            f"  {experiment:<16}{model:<22}"
            f"{len(ordered_catalog_rows):>2} member(s)  READY"
        )

print(
    f"\nReady: {len(maria_tas_tasks)} model/experiment combination(s); "
    f"problems: {len(tas_inventory_problems)}"
)

# Do not silently create tas for only part of an N ensemble.
if tas_inventory_problems:
    raise RuntimeError(
        "Some completed N ensembles do not have matching Maria tas. "
        "Review the inventory above before processing."
    )

Maria original tas matched to completed g025 N outputs:
  amip-hist       BCC-CSM2-MR            1 member(s)  READY
  amip-hist       CAMS-CSM1-0            3 member(s)  READY
  amip-hist       CESM2                  3 member(s)  READY
  amip-piForcing  CESM2                  1 member(s)  READY
  amip-hist       CIESM                  3 member(s)  READY
  amip-hist       CNRM-CM6-1            10 member(s)  READY
  amip-piForcing  CNRM-CM6-1             1 member(s)  READY
  amip-hist       CNRM-CM6-1-HR          1 member(s)  READY
  amip-hist       CNRM-ESM2-1            1 member(s)  READY
  amip-hist       CanESM5               10 member(s)  READY
  amip-piForcing  CanESM5                3 member(s)  READY
  amip-hist       FGOALS-f3-L            3 member(s)  READY
  amip-hist       FGOALS-g3              4 member(s)  READY
  amip-hist       FIO-ESM-2-0            3 member(s)  READY
  amip-piForcing  HadGEM3-GC31-LL        1 member(s)  READY
  amip-hist       IITM-ESM               1 m

In [17]:
def create_member_tas(row, temporary_tas_file):
    """
    Merge Maria's original tas time chunks, select 1870–2014, and regrid
    the member to the same g025 target grid used for N.
    """
    tas_paths = maria_tas_paths(row)

    temporary_input_directory = Path(
        tempfile.mkdtemp(prefix="cdo_tas_inputs_")
    )

    try:
        # A single original file can be passed directly to the regrid step.
        if len(tas_paths) == 1:
            merged_tas = tas_paths[0]

        else:
            merged_tas = (
                temporary_input_directory / "tas_merged.nc"
            )

            merge_command = [
                "cdo",
                "-L",
                "-O",
                "mergetime",
                *map(str, tas_paths),
                str(merged_tas),
            ]

            merge_result = subprocess.run(
                merge_command,
                text=True,
                capture_output=True,
            )

            if merge_result.returncode != 0:
                print("\nCDO command:")
                print(" ".join(merge_command))
                print("\nCDO stderr:")
                print(merge_result.stderr)

                raise RuntimeError(
                    f"CDO tas mergetime failed for "
                    f"{row['model']} / {row['experiment']} / "
                    f"{row['member']}"
                )

        regrid_command = [
            "cdo",
            "-L",
            "-O",
            "-f", "nc4c",
            "-z", "zip_4",
            f"{tas_remap_operator},{g025_template}",
            f"-seldate,{start_date},{end_date}",
            "-selname,tas",
            str(merged_tas),
            str(temporary_tas_file),
        ]

        print(f"    CDO: {row['member']}")

        regrid_result = subprocess.run(
            regrid_command,
            text=True,
            capture_output=True,
        )

        if regrid_result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(regrid_command))
            print("\nCDO stderr:")
            print(regrid_result.stderr)

            raise RuntimeError(
                f"CDO tas regridding failed for "
                f"{row['model']} / {row['experiment']} / "
                f"{row['member']}"
            )

    finally:
        shutil.rmtree(
            temporary_input_directory,
            ignore_errors=True,
        )

In [18]:
def copy_tas_member_to_combined(
    source_file,
    combined_file,
    member_index,
    member_name,
):
    """
    Append one regridded Maria tas member to a compressed combined file.

    The result contains:
      member(member): exact CMIP6 labels copied from N
      tas(member, time, lat, lon)

    Data are copied in 12-month blocks to keep memory use small.
    """
    with Dataset(source_file, "r") as source:
        if "tas" not in source.variables:
            raise KeyError(
                f"No tas variable in temporary file {source_file}"
            )

        source_tas = source.variables["tas"]
        source_dimensions = source_tas.dimensions

        if "time" not in source_dimensions:
            raise ValueError(
                f"Unexpected tas dimensions: {source_dimensions}"
            )

        if not combined_file.exists():
            with Dataset(
                combined_file,
                "w",
                format="NETCDF4",
            ) as destination:
                destination.createDimension("member", None)

                for dimension in source_dimensions:
                    destination.createDimension(
                        dimension,
                        len(source.dimensions[dimension]),
                    )

                # Copy the one-dimensional time, latitude, and longitude
                # coordinate variables created by CDO.
                for dimension in source_dimensions:
                    if dimension not in source.variables:
                        continue

                    source_coordinate = source.variables[dimension]

                    if source_coordinate.dimensions != (dimension,):
                        continue

                    coordinate_options = {}

                    if "_FillValue" in source_coordinate.ncattrs():
                        coordinate_options["fill_value"] = (
                            source_coordinate.getncattr("_FillValue")
                        )

                    destination_coordinate = (
                        destination.createVariable(
                            dimension,
                            source_coordinate.dtype,
                            (dimension,),
                            **coordinate_options,
                        )
                    )

                    destination_coordinate[:] = (
                        source_coordinate[:]
                    )

                    for attribute in source_coordinate.ncattrs():
                        if attribute != "_FillValue":
                            destination_coordinate.setncattr(
                                attribute,
                                source_coordinate.getncattr(
                                    attribute
                                ),
                            )

                member_coordinate = destination.createVariable(
                    "member",
                    str,
                    ("member",),
                )
                member_coordinate.long_name = (
                    "CMIP6 ensemble member identifier"
                )
                member_coordinate.comment = (
                    "Labels and ordering copied from the matching "
                    "processed g025 N file."
                )

                chunk_sizes = [1]

                for dimension in source_dimensions:
                    if dimension == "time":
                        chunk_sizes.append(
                            min(
                                12,
                                len(source.dimensions[dimension]),
                            )
                        )
                    else:
                        chunk_sizes.append(
                            len(source.dimensions[dimension])
                        )

                fill_value = np.float32(
                    getattr(source_tas, "_FillValue", np.nan)
                )

                tas_output = destination.createVariable(
                    "tas",
                    "f4",
                    ("member",) + source_dimensions,
                    zlib=True,
                    complevel=4,
                    chunksizes=tuple(chunk_sizes),
                    fill_value=fill_value,
                )

                for attribute in source_tas.ncattrs():
                    if attribute != "_FillValue":
                        tas_output.setncattr(
                            attribute,
                            source_tas.getncattr(attribute),
                        )

                tas_output.regridding = (
                    f"CDO {tas_remap_operator} to g025"
                )
                tas_output.source = (
                    "Original Maria MMLEA model output"
                )

        with Dataset(combined_file, "a") as destination:
            destination.variables["member"][
                member_index
            ] = member_name

            destination_tas = destination.variables["tas"]

            source_time_axis = source_dimensions.index("time")
            destination_time_axis = source_time_axis + 1
            number_of_months = len(source.dimensions["time"])

            # Write one year at a time.
            for time_start in range(0, number_of_months, 12):
                time_stop = min(
                    time_start + 12,
                    number_of_months,
                )

                source_index = [
                    slice(None)
                    for _ in source_dimensions
                ]
                source_index[source_time_axis] = slice(
                    time_start,
                    time_stop,
                )

                destination_index = [
                    slice(None)
                    for _ in destination_tas.dimensions
                ]
                destination_index[0] = member_index
                destination_index[
                    destination_time_axis
                ] = slice(
                    time_start,
                    time_stop,
                )

                destination_tas[
                    tuple(destination_index)
                ] = source_tas[
                    tuple(source_index)
                ]

In [19]:
for task in maria_tas_tasks:
    model = task["model"]
    experiment = task["experiment"]
    catalog_rows = task["catalog_rows"]

    model_output_directory = output_root / model
    model_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    final_file = (
        model_output_directory
        / (
            f"tas_Amon_{model}_{experiment}_combined_"
            f"{output_period}_g025.nc"
        )
    )

    partial_file = final_file.with_suffix(".partial.nc")

    if final_file.exists() and not tas_overwrite:
        print(f"Skipping existing file: {final_file}")
        continue

    if partial_file.exists():
        partial_file.unlink()

    print(
        f"\n{model} / {experiment}: "
        f"{len(catalog_rows)} member(s)"
    )

    try:
        with tempfile.TemporaryDirectory(
            prefix="cdo_tas_"
        ) as temporary_directory:
            temporary_directory = Path(temporary_directory)

            for member_index, row in enumerate(catalog_rows):
                temporary_tas_file = (
                    temporary_directory
                    / f"{row['member']}_tas.nc"
                )

                create_member_tas(
                    row,
                    temporary_tas_file,
                )

                copy_tas_member_to_combined(
                    temporary_tas_file,
                    partial_file,
                    member_index,
                    row["member"],
                )

                temporary_tas_file.unlink()

        partial_file.replace(final_file)

        print(f"  Saved: {final_file}")
        subprocess.run(
            ["du", "-h", str(final_file)],
            check=True,
        )

    except Exception:
        print(
            f"  FAILED: incomplete output retained at "
            f"{partial_file}"
        )
        raise


BCC-CSM2-MR / amip-hist: 1 member(s)
    CDO: r1i1p1f1
  Saved: data/AMIP/BCC-CSM2-MR/tas_Amon_BCC-CSM2-MR_amip-hist_combined_187001-201412_g025.nc

CAMS-CSM1-0 / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CAMS-CSM1-0/tas_Amon_CAMS-CSM1-0_amip-hist_combined_187001-201412_g025.nc

CESM2 / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CESM2/tas_Amon_CESM2_amip-hist_combined_187001-201412_g025.nc

CESM2 / amip-piForcing: 1 member(s)
    CDO: r1i1p1f1
  Saved: data/AMIP/CESM2/tas_Amon_CESM2_amip-piForcing_combined_187001-201412_g025.nc

CIESM / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CIESM/tas_Amon_CIESM_amip-hist_combined_187001-201412_g025.nc

CNRM-CM6-1 / amip-hist: 10 member(s)
    CDO: r1i1p1f2
    CDO: r2i1p1f2
    CDO: r3i1p1f2
    CDO: r4i1p1f2
    CDO: r5i1p1f2
    CDO: r6i1p1f2
    CDO: r7i1p1f2
    CDO: r8i1p1f2
    CDO: 

In [ ]:
def netcdf_month_sequence(dataset):
    """Represent a NetCDF time coordinate as sequential YYYYMM numbers."""
    time = dataset.variables["time"]
    calendar = getattr(time, "calendar", "standard")

    dates = num2date(
        time[:],
        units=time.units,
        calendar=calendar,
    )

    return tuple(
        month_number(f"{date.year:04d}{date.month:02d}")
        for date in dates
    )


expected_month_sequence = tuple(
    range(
        month_number("187001"),
        month_number("201412") + 1,
    )
)

verification_failures = []

for task in maria_tas_tasks:
    model = task["model"]
    experiment = task["experiment"]
    n_file = task["n_file"]

    tas_file = (
        output_root
        / model
        / (
            f"tas_Amon_{model}_{experiment}_combined_"
            f"{output_period}_g025.nc"
        )
    )

    issues = []

    if not tas_file.exists():
        issues.append("tas output is missing")

    else:
        with (
            Dataset(n_file, "r") as n_dataset,
            Dataset(tas_file, "r") as tas_dataset,
        ):
            n_members = [
                decode_member_label(value)
                for value in n_dataset.variables["member"][:]
            ]
            tas_members = [
                decode_member_label(value)
                for value in tas_dataset.variables["member"][:]
            ]

            if n_members != tas_members:
                issues.append(
                    "member labels or ordering differ"
                )

            n_shape = n_dataset.variables["N"].shape
            tas_shape = tas_dataset.variables["tas"].shape

            if n_shape != tas_shape:
                issues.append(
                    f"data shapes differ: N={n_shape}, tas={tas_shape}"
                )

            for coordinate in ("lat", "lon"):
                if (
                    coordinate not in n_dataset.variables
                    or coordinate not in tas_dataset.variables
                ):
                    issues.append(
                        f"missing {coordinate} coordinate"
                    )
                    continue

                if not np.allclose(
                    n_dataset.variables[coordinate][:],
                    tas_dataset.variables[coordinate][:],
                    equal_nan=True,
                ):
                    issues.append(
                        f"{coordinate} grids differ"
                    )

            if (
                netcdf_month_sequence(n_dataset)
                != expected_month_sequence
            ):
                issues.append(
                    "N time coverage is not exactly 1870-01–2014-12"
                )

            if (
                netcdf_month_sequence(tas_dataset)
                != expected_month_sequence
            ):
                issues.append(
                    "tas time coverage is not exactly 1870-01–2014-12"
                )

    if issues:
        verification_failures.append(
            (model, experiment, issues)
        )
        print(
            f"PROBLEM  {experiment:<16}{model:<22}"
            + "; ".join(issues)
        )
    else:
        print(
            f"OK       {experiment:<16}{model:<22}"
            f"{len(task['member_labels'])} member(s)"
        )

if verification_failures:
    raise RuntimeError(
        f"{len(verification_failures)} tas/N verification(s) failed"
    )

print(
    "\nAll Maria-derived tas files match their corresponding g025 N files."
)